# 🚀 Amazing-QR: Tam Otomatik (CSV) QR Oluşturucu

Bu notebook, **Amazing-QR** projesini tamamen CSV (sipariş listesi) odaklı çalıştırmak için optimize edilmiştir.

---

## 🛠️ 1. Kurulum ve Hazırlık


In [ ]:
# Projeyi klonla
!git clone https://github.com/orioninsist/amazing-qr.git
%cd amazing-qr

# Gereksinimleri yükle
!pip install -r requirements.txt
!pip install . # amzqr paketini sisteme yükle

# Klasör yapısını hazırla
!mkdir -p output


## 📊 2. Sipariş Listesini Görüntüle


In [ ]:
import pandas as pd
import os
from IPython.display import display, HTML

csv_path = 'inputs/order.csv'

if os.path.exists(csv_path):
    try:
        df = pd.read_csv(csv_path)
        display(HTML("<h3 style='color: #2c3e50;'>📋 Mevcut Sipariş Listesi</h3>"))
        display(df)
    except Exception as e:
        print(f"❌ CSV okuma hatası: {e}")
else:
    display(HTML("<div style='padding: 20px; background: #fff4e5; border-left: 5px solid #ffa117; border-radius: 4px;'>⚠️ inputs/order.csv bulunamadı.</div>"))


## ✍️ 3. Siparişleri Düzenle (Editor)


In [ ]:
import pandas as pd
import os
import re
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

csv_path = 'inputs/order.csv'

def slugify(value):
    value = re.sub(r'^https?://', '', str(value))
    value = re.sub(r'[^\w\s-]', '_', value).strip().lower()
    return re.sub(r'[-\s]+', '_', value)[:50]

def load_data():
    if not os.path.exists(csv_path):
        return pd.DataFrame(columns=['words', 'version', 'level', 'picture', 'colorized', 'contrast', 'brightness', 'save_name'])
    
    try:
        df = pd.read_csv(csv_path)
    except:
        return pd.DataFrame(columns=['words', 'version', 'level', 'picture', 'colorized', 'contrast', 'brightness', 'save_name'])
        
    cols = ['words', 'version', 'level', 'picture', 'colorized', 'contrast', 'brightness', 'save_name']
    for c in cols:
        if c not in df.columns: df[c] = None
    
    df['version'] = pd.to_numeric(df['version'], errors='coerce').fillna(1).astype(int)
    df['level'] = df['level'].fillna('H')
    df['contrast'] = pd.to_numeric(df['contrast'], errors='coerce').fillna(1.0).astype(float)
    df['brightness'] = pd.to_numeric(df['brightness'], errors='coerce').fillna(1.0).astype(float)
    
    for i, row in df.iterrows():
        if pd.isna(row['colorized']) or str(row['colorized']).strip() in ['', 'nan', 'None']:
            df.at[i, 'colorized'] = True if pd.notna(row.get('picture')) and str(row.get('picture')).strip() not in ['', 'nan', 'None'] else False
        else:
            df.at[i, 'colorized'] = str(row['colorized']).lower() == 'true'
            
        if pd.isna(row['save_name']) or str(row['save_name']).strip() in ['', 'nan', 'None']:
            ext = '.gif' if str(row.get('picture', '')).lower().endswith('.gif') else '.png'
            slug = slugify(row['words']) or f"qr_{i+1}"
            df.at[i, 'save_name'] = f"{slug}{ext}"
            
    return df[cols]

def save_data(df):
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df.to_csv(csv_path, index=False)

# UI Logic
output = widgets.Output()

def refresh_view():
    with output:
        clear_output()
        df = load_data()
        
        if df.empty:
            display(HTML("<div style='padding: 20px; background: #fff4e5; border-left: 5px solid #ffa117; border-radius: 4px; margin: 10px 0;'>⚠️ Sipariş listesi boş.</div>"))
        else:
            header_html = """
            <div style="display: grid; grid-template-columns: 200px 80px 80px 120px 80px 80px 80px 160px; gap: 5px; background: #2c3e50; color: white; padding: 10px; border-radius: 8px 8px 0 0; font-weight: bold; font-family: sans-serif; font-size: 12px; align-items: center;">
                <div>URL / Metin</div>
                <div style='text-align:center;'>Ver</div>
                <div style='text-align:center;'>Lvl</div>
                <div>Görsel (Asset)</div>
                <div style='text-align:center;'>Color</div>
                <div style='text-align:center;'>Cont</div>
                <div style='text-align:center;'>Bright</div>
                <div style='text-align:center;'>İşlemler</div>
            </div>
            """
            display(HTML(header_html))
            
            rows = []
            for i, row in df.iterrows():
                edit_btn = widgets.Button(description="Düzenle", button_style='info', icon='pencil', layout={'width': '80px', 'height': '30px'})
                delete_btn = widgets.Button(description="Sil", button_style='danger', icon='trash', layout={'width': '60px', 'height': '30px'})
                
                edit_btn.on_click(lambda b, idx=i: show_editor(idx))
                delete_btn.on_click(lambda b, idx=i: delete_row(idx))
                
                row_widget = widgets.HBox([
                    widgets.Label(str(row['words']), layout={'width': '200px'}),
                    widgets.Label(str(row['version']), layout={'width': '80px'}, style={'text_align': 'center'}),
                    widgets.Label(str(row['level']), layout={'width': '80px'}),
                    widgets.Label(str(row['picture']) if pd.notna(row['picture']) and str(row['picture']) != 'nan' else "-", layout={'width': '120px'}),
                    widgets.Label("✅" if bool(row['colorized']) else "❌", layout={'width': '80px'}),
                    widgets.Label(f"{float(row['contrast']):.1f}", layout={'width': '80px'}),
                    widgets.Label(f"{float(row['brightness']):.1f}", layout={'width': '80px'}),
                    widgets.HBox([edit_btn, delete_btn], layout={'width': '160px', 'gap': '5px', 'justify_content': 'center'})
                ], layout={'border-bottom': '1px solid #eee', 'padding': '5px', 'align_items': 'center', 'font_family': 'sans-serif', 'font_size': '12px'})
                rows.append(row_widget)
            
            display(widgets.VBox(rows, layout={'border': '1px solid #eee', 'border-top': 'none', 'border-radius': '0 0 8px 8px', 'background': 'white', 'padding': '0'}))
            
        add_btn = widgets.Button(description="➕ Yeni Sipariş", button_style='success', layout={'width': '150px', 'height': '40px'})
        bulk_btn = widgets.Button(description="🛠️ Toplu Düzenle", button_style='warning', layout={'width': '150px', 'height': '40px'})
        
        add_btn.on_click(lambda b: show_editor(None))
        bulk_btn.on_click(lambda b: show_bulk_editor())
        
        display(widgets.HBox([add_btn, bulk_btn], layout={'margin': '20px 0', 'gap': '10px'}))

def delete_row(idx):
    df = load_data()
    df = df.drop(idx).reset_index(drop=True)
    save_data(df)
    refresh_view()

def show_editor(idx=None):
    df = load_data()
    if idx is not None:
        item = df.iloc[idx]
    else:
        item = {'words': '', 'version': 1, 'level': 'H', 'picture': '', 'colorized': True, 'contrast': 1.0, 'brightness': 1.0, 'save_name': ''}
    
    words_input = widgets.Text(value=str(item['words']), description='<b>URL/Metin:</b>', placeholder='https://...', style={'description_width': '100px'}, layout={'width': '100%'})
    picture_input = widgets.Text(value=str(item['picture']) if pd.notna(item['picture']) and str(item['picture']) != 'nan' else '', description='<b>Görsel Adı:</b>', placeholder='logo.png (Opsiyonel)', style={'description_width': '100px'}, layout={'width': '100%'})
    version_input = widgets.IntSlider(value=int(item['version']), min=1, max=40, step=1, description='Versiyon:', style={'description_width': '100px'})
    level_input = widgets.Dropdown(options=['L', 'M', 'Q', 'H'], value=str(item['level']), description='Hata Payı:', style={'description_width': '100px'})
    colorized_input = widgets.Checkbox(value=bool(item['colorized']), description='Renklendir (Colorized)', style={'description_width': '100px'})
    contrast_input = widgets.FloatSlider(value=float(item['contrast']), min=0.1, max=3.0, step=0.1, description='Kontrast:', style={'description_width': '100px'})
    brightness_input = widgets.FloatSlider(value=float(item['brightness']), min=0.1, max=3.0, step=0.1, description='Parlaklık:', style={'description_width': '100px'})
    save_name_input = widgets.Text(value=str(item['save_name']), description='<b>Kayıt Adı:</b>', placeholder='auto_generated.png', style={'description_width': '100px'}, layout={'width': '100%'})

    save_btn = widgets.Button(description="Kaydet", button_style='primary', icon='check', layout={'width': '120px'})
    cancel_btn = widgets.Button(description="İptal", button_style='', icon='times', layout={'width': '100px'})
    
    def on_save(b):
        new_row = {
            'words': words_input.value,
            'version': version_input.value,
            'level': level_input.value,
            'picture': picture_input.value,
            'colorized': colorized_input.value,
            'contrast': contrast_input.value,
            'brightness': brightness_input.value,
            'save_name': save_name_input.value
        }
        
        current_df = load_data()
        if idx is not None:
            for col, val in new_row.items():
                current_df.at[idx, col] = val
        else:
            current_df = pd.concat([current_df, pd.DataFrame([new_row])], ignore_index=True)
            
        save_data(current_df)
        refresh_view()

    def on_cancel(b):
        refresh_view()

    save_btn.on_click(on_save)
    cancel_btn.on_click(on_cancel)
    
    with output:
        clear_output()
        title = "✍️ Siparişi Düzenle" if idx is not None else "➕ Yeni Sipariş Ekle"
        display(HTML(f"<h3 style='color: #2980b9; margin-bottom: 20px;'>{title}</h3>"))
        
        form = widgets.VBox([
            words_input, 
            picture_input, 
            widgets.HBox([version_input, level_input]),
            colorized_input,
            widgets.HBox([contrast_input, brightness_input]),
            save_name_input,
            widgets.HBox([save_btn, cancel_btn], layout={'margin': '30px 0 10px 0', 'gap': '15px'})
        ], layout={'padding': '30px', 'border': '1px solid #3498db', 'border-radius': '15px', 'background': '#fdfdfd', 'box-shadow': '0 4px 15px rgba(0,0,0,0.05)'})
        display(form)

def show_bulk_editor():
    df = load_data()
    if df.empty:
        refresh_view()
        return

    version_check = widgets.Checkbox(value=False, description='Versiyonu Değiştir', layout={'width': '200px'})
    version_input = widgets.IntSlider(value=40, min=1, max=40, step=1, layout={'width': '300px'})
    
    level_check = widgets.Checkbox(value=False, description='Hata Payını Değiştir', layout={'width': '200px'})
    level_input = widgets.Dropdown(options=['L', 'M', 'Q', 'H'], value='H', layout={'width': '100px'})
    
    picture_check = widgets.Checkbox(value=False, description='Görseli Değiştir', layout={'width': '200px'})
    picture_input = widgets.Text(value='', placeholder='logo.png', layout={'width': '300px'})
    
    color_check = widgets.Checkbox(value=False, description='Rengi Değiştir', layout={'width': '200px'})
    color_input = widgets.Checkbox(value=True, description='Renklendir (Colorized)')
    
    contrast_check = widgets.Checkbox(value=False, description='Kontrastı Değiştir', layout={'width': '200px'})
    contrast_input = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, layout={'width': '300px'})
    
    brightness_check = widgets.Checkbox(value=False, description='Parlaklığı Değiştir', layout={'width': '200px'})
    brightness_input = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, layout={'width': '300px'})

    apply_btn = widgets.Button(description="Tümüne Uygula", button_style='warning', icon='bolt', layout={'width': '180px', 'height': '40px'})
    cancel_btn = widgets.Button(description="İptal", icon='times', layout={'width': '100px', 'height': '40px'})

    def on_apply(b):
        current_df = load_data()
        for i in range(len(current_df)):
            if version_check.value: current_df.at[i, 'version'] = version_input.value
            if level_check.value: current_df.at[i, 'level'] = level_input.value
            if picture_check.value: current_df.at[i, 'picture'] = picture_input.value
            if color_check.value: current_df.at[i, 'colorized'] = color_input.value
            if contrast_check.value: current_df.at[i, 'contrast'] = contrast_input.value
            if brightness_check.value: current_df.at[i, 'brightness'] = brightness_input.value
        
        save_data(current_df)
        refresh_view()

    def on_cancel(b):
        refresh_view()

    apply_btn.on_click(on_apply)
    cancel_btn.on_click(on_cancel)

    with output:
        clear_output()
        display(HTML("<h3 style='color: #e67e22; margin-bottom: 20px;'>🛠️ Toplu Düzenleme Paneli</h3>"))
        display(HTML("<p style='color: #7f8c8d; margin-bottom: 20px;'>Hangi alanları tüm listeye uygulamak istiyorsanız seçin ve değeri ayarlayın.</p>"))
        
        bulk_form = widgets.VBox([
            widgets.HBox([version_check, version_input]),
            widgets.HBox([level_check, level_input]),
            widgets.HBox([picture_check, picture_input]),
            widgets.HBox([color_check, color_input]),
            widgets.HBox([contrast_check, contrast_input]),
            widgets.HBox([brightness_check, brightness_input]),
            widgets.HBox([apply_btn, cancel_btn], layout={'margin': '30px 0 10px 0', 'gap': '15px'})
        ], layout={'padding': '30px', 'border': '1px solid #e67e22', 'border-radius': '15px', 'background': '#fffaf5', 'box-shadow': '0 4px 15px rgba(0,0,0,0.05)'})
        display(bulk_form)

refresh_view()
display(output)


## 🚀 4. Toplu İşlemi Başlat


In [ ]:
!python batch_process.py


## ✨ 5. Sonuçları Önizle


In [ ]:
import os
import base64
from IPython.display import display, HTML

output_dir = 'output'
if os.path.exists(output_dir):
    files = sorted([f for f in os.listdir(output_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif'))])
    if not files:
        print("❌ Üretilen dosya bulunamadı.")
    else:
        html_content = '<div style="display: flex; flex-wrap: wrap; gap: 25px; justify-content: center; padding: 30px; background: #f0f2f5; border-radius: 20px;">'
        for f in files:
            f_path = os.path.join(output_dir, f)
            try:
                with open(f_path, "rb") as img_file:
                    encoded_string = base64.b64encode(img_file.read()).decode('utf-8')
                
                mime_type = "image/gif" if f.lower().endswith(".gif") else "image/png"
                img_data = f"data:{mime_type};base64,{encoded_string}"
                
                html_content += f'''
                <div style="text-align: center; background: white; padding: 15px; border-radius: 15px; box-shadow: 0 10px 20px rgba(0,0,0,0.1); width: 230px;">
                    <img src="{img_data}" style="width: 200px; height: 200px; object-fit: contain; border-radius: 10px; margin-bottom: 10px;">
                    <p style="font-size: 13px; font-weight: bold; color: #333; word-wrap: break-word; margin: 0; font-family: sans-serif;">{f}</p>
                </div>
                '''
            except Exception as e:
                print(f"⚠️ {f} yüklenemedi: {e}")
                
        html_content += '</div>'
        display(HTML(html_content))


## 📥 6. Sonuçları İndir (ZIP)


In [ ]:
import shutil
from google.colab import files
import datetime

now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f'amazing_qr_results_{now}'

try:
    shutil.make_archive(zip_name, 'zip', 'output')
    files.download(f'{zip_name}.zip')
    print(f"✅ {zip_name}.zip indiriliyor...")
except Exception as e:
    print(f"❌ Hata: {e}")
